# Классификация ботов по логам событий
## Постановка задачи
Даны логи событий на маркетплейсе. Для каждого `cookie_id` (сессии) нужно предсказать
вероятность того, что это бот (`target=1`).

**Метрика:** Precision@Recall=0.7. То есть максимизируем Precision при условии, что
Recall >= 0.7.

**Ключевое ограничение:** признаки можно считать **только** по событиям внутри окна
`[window_start_ts, window_end_ts)` для соответствующей куки. Никаких агрегатов по всей
истории — это утечка из будущего.

## Подход
1. Отфильтровать события по окну **до** любых временных вычислений.
2. Построить поведенческие фичи по трём группам:
   - **UA-фичи** — явные признаки ботов (`python`, `go-http`), парсинг ОС/браузера,
     разнообразие User-Agent.
   - **Временные фичи** — `time_delta` между событиями внутри окна (mean, std, min),
     доля «резких» переходов.
   - **Pointer-фичи** — статистики координат курсора.
   - **Event/item-фичи** — доли событий каждого типа, разнообразие категорий/локаций,
     конверсия в контактные действия.
3. Модель: CatBoost с нативными категориальными фичами.
4. Валидация: временной сплит (train до 2026-04-17, val после)

In [1]:
!pip install user-agents catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 3.8 MB/s eta 0:00:00


### **random seed** для всех ячеек

In [2]:
rs = 0

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **Считываем данные**

In [4]:

import numpy as np
import pandas as pd

DATA_DIR = '/content/drive/MyDrive/bot_detection_challenge/data/'

train = pd.read_csv(DATA_DIR + 'train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv(DATA_DIR + 'test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv(DATA_DIR + 'events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [5]:
# ВАЖНО: убедиться, что каждая кука встречается только в одном окне.
# Если это не так — нельзя мержить агрегаты только по cookie_id, будет утечка.
print('Уникальных cookie_id в train:', train.cookie_id.nunique())
print('Строк в train:                   ', len(train))

print('Уникальных cookie_id в test:', test.cookie_id.nunique())
print('Строк в test:                   ', len(test))

assert train.cookie_id.nunique() == len(train), \
    'Одна кука встречается в нескольких окнах — нужно группировать по трём ключам!'

assert test.cookie_id.nunique() == len(test), \
    'Одна кука встречается в нескольких окнах — нужно группировать по трём ключам!'

Уникальных cookie_id в train: 11091
Строк в train:                    11091
Уникальных cookie_id в test: 4909
Строк в test:                    4909


In [6]:
# Убеждаемся, что время в формате datetime
events['event_ts'] = pd.to_datetime(events['event_ts'])
#Приводим platform к нижнему регистру, чтобы не держать одни и те же категории, но в разно регистре
def rename(s):
  s = s.lower()
  return s
events['platform'] = events['platform'].apply(rename)
events['platform']

,platform
0,desktop
1,web
2,web
3,web
4,web
...,...
328900,android
328901,desktop
328902,android
328903,android


## **Создаём фичи для нашего датасета**

### Здесь работаем на уровне событий с user_agent колонкой.

In [7]:
from user_agents import parse

# Сначала извлекаем признаки на уровне СОБЫТИЙ
def extract_ua_features(df):
    df = df.copy()
    df['ua_lower'] = df['user_agent'].str.lower()

    # Явные боты
    df['ua_is_python'] = df['ua_lower'].str.contains('python|urllib|requests', regex=True).astype(int)
    df['ua_is_go'] = df['ua_lower'].str.contains('go-http', regex=True).astype(int)
    df['ua_is_curl'] = df['ua_lower'].str.contains('curl|wget', regex=True).astype(int)

    # Парсинг
    parsed = df['user_agent'].apply(lambda x: parse(str(x)))
    df['ua_os'] = parsed.apply(lambda x: x.os.family)
    df['ua_browser'] = parsed.apply(lambda x: x.browser.family)
    df['ua_is_mobile'] = parsed.apply(lambda x: int(x.is_mobile))
    df['ua_is_pc'] = parsed.apply(lambda x: int(x.is_pc))

    return df

events = extract_ua_features(events)


### Фильтрация по временному окну

In [8]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

198436 89690


### Добавляем разницу по времени между событиями


In [9]:
def add_time_delta(ev):
    ev = ev.sort_values(['cookie_id', 'window_start_ts', 'event_ts']).copy()

    ev['time_delta_sec'] = (
        ev.groupby(['cookie_id'])['event_ts']
          .diff()
          .dt.total_seconds()
    )

    return ev

ev_tr = add_time_delta(ev_tr)
ev_te = add_time_delta(ev_te)

### Собираем статистику для времени между событиями. У ботов время между событиями короче чем у людей.

In [10]:

def add_time_features(ev):
  time_features = ev.groupby(
      'cookie_id'
  )['time_delta_sec'].agg(
      time_mean='mean', time_max='max', time_min='min',
      time_std='std', time_median='median', n_events='size'
  ).reset_index()

  time_cols = ['time_mean', 'time_max', 'time_min', 'time_std', 'time_median']
  time_features[time_cols] = time_features[time_cols].fillna(-1)
  return time_features

t_f_tr = add_time_features(ev_tr)
train = train.merge(t_f_tr, on='cookie_id', how='left')

t_f_te = add_time_features(ev_te)
test = test.merge(t_f_te, on='cookie_id', how='left')


### Pointer фичи.

In [11]:
def add_pointer_stats(ev):
  pointer_x_features = ev.groupby(
      'cookie_id'
  )['pointer_x'].agg(
    pointer_x_mean='mean', pointer_x_max='max', pointer_x_min='min',
    pointer_x_std='std', pointer_x_median='median'
  ).reset_index()
  pointer_y_features = ev.groupby(
      'cookie_id'
  )['pointer_y'].agg(
    pointer_y_mean='mean', pointer_y_max='max', pointer_y_min='min',
    pointer_y_std='std', pointer_y_median='median'
  ).reset_index()
  return pd.merge(pointer_x_features, pointer_y_features, on='cookie_id')
pointer_st_tr = add_pointer_stats(ev_tr)
pointer_st_tr = pointer_st_tr.fillna(-1)
pointer_st_tr.columns

Index(['cookie_id', 'pointer_x_mean', 'pointer_x_max', 'pointer_x_min',
       'pointer_x_std', 'pointer_x_median', 'pointer_y_mean', 'pointer_y_max',
       'pointer_y_min', 'pointer_y_std', 'pointer_y_median'],
      dtype='object')

In [12]:
train = train.merge(pointer_st_tr, on = 'cookie_id', how='left')

In [13]:
pointer_st_te = add_pointer_stats(ev_te)
pointer_st_te = pointer_st_te.fillna(-1)
test = test.merge(pointer_st_te, on = 'cookie_id', how='left')

In [14]:
def pointer_dynamics(ev):
    """
    Динамика курсора: не координаты, а перемещения.
    Требует сортировки по event_ts внутри окна.
    """
    ev = ev.sort_values(['cookie_id', 'event_ts']).copy()
    g = ev.groupby('cookie_id')
    ev['dx'] = g['pointer_x'].diff()
    ev['dy'] = g['pointer_y'].diff()
    ev['dist'] = np.sqrt(ev['dx']**2 + ev['dy']**2)

    out = ev.groupby('cookie_id').agg(
        pointer_total_distance =('dist', 'sum'),
        pointer_max_jump       =('dist', 'max'),
        pointer_zero_move_ratio=('dist', lambda s: (s == 0).mean()),
    )
    return out.reset_index()

ptr_tr = pointer_dynamics(ev_tr)
ptr_te = pointer_dynamics(ev_te)
train = train.merge(ptr_tr, on='cookie_id', how='left')
test = test.merge(ptr_te, on='cookie_id', how='left')

### User agent признаки.

In [15]:

def user_agent_features(ev):
  agg_features = ev.groupby('cookie_id').agg(
      # Счетчики
      n_unique_ua=('user_agent', 'nunique'),
      n_unique_query=('search_query', 'nunique'),

      # Флаги ботов (Max)
      has_python_ua=('ua_is_python', 'max'),
      has_go_ua=('ua_is_go', 'max'),
      has_curl_ua=('ua_is_curl', 'max'),

      #search page max, обычно пользователи не уходят дальше 3 страницы
      max_search=('search_page', 'max'),

      # Пропорции (Mean)
      mobile_ratio=('ua_is_mobile', 'mean'),
      pc_ratio=('ua_is_pc', 'mean'),
      mean_search=('search_page', 'mean'),

      # Категориальные (Mode)
      main_platform = ('platform', lambda x: x.mode()[0] if not x.mode().empty else 'unknown'),
      main_os=('ua_os', lambda x: x.mode()[0] if not x.mode().empty else 'unknown'),
      main_browser=('ua_browser', lambda x: x.mode()[0] if not x.mode().empty else 'unknown')
  ).reset_index()
  return agg_features


In [16]:
ua_features_tr = user_agent_features(ev_tr)
ua_features_te = user_agent_features(ev_te)
train = train.merge(ua_features_tr, on = 'cookie_id', how='left')
test = test.merge(ua_features_te, on='cookie_id', how='left')

### Добавим фичи связанные с event_name

In [17]:
def add_event_features(ev):

    # 1. Бинарные флаги: был ли event_name в окне
    binary = pd.crosstab(
        ev['cookie_id'],
        ev['event_name']
    )
    binary = (binary > 0).astype(np.int8)
    binary.columns = [f'ev_{c}' for c in binary.columns]  # префикс, чтобы не конфликтовало
    binary = binary.reset_index()

    # 2. Число уникальных event_name в окне
    n_unique = ev.groupby('cookie_id')['event_name'].nunique().rename('n_unique_events').reset_index()

    return binary.merge(n_unique, on='cookie_id')

event_features_tr = add_event_features(ev_tr)
event_features_te = add_event_features(ev_te)

In [18]:
train = train.merge(event_features_tr, on='cookie_id', how='left')
test = test.merge(event_features_te, on='cookie_id', how='left')

### Добавляем фичи связанные с items.

In [19]:
def add_item_features(df, ev):
  df['n_unique_items'] = ev.groupby('cookie_id')['item_id'].nunique()
  df['item_diversity'] = df['n_unique_items'] / df['n_events']

  df['n_unique_categories'] = ev.groupby('cookie_id')['item_category'].nunique()
  df['n_unique_locations']  = ev.groupby('cookie_id')['item_location'].nunique()
  df['n_unique_sellers']    = ev.groupby('cookie_id')['seller_type'].nunique()
  return df
train = add_item_features(train, ev_tr)
test = add_item_features(test, ev_te)

In [20]:
ev_counts = ev_tr.groupby('cookie_id')['event_name'].value_counts().unstack(fill_value=0)
ev_ratios = ev_counts.div(ev_counts.sum(axis=1), axis=0).add_prefix('ratio_')
train = train.merge(ev_ratios, on='cookie_id', how='left')

ev_counts = ev_te.groupby('cookie_id')['event_name'].value_counts().unstack(fill_value=0)
ev_ratios = ev_counts.div(ev_counts.sum(axis=1), axis=0).add_prefix('ratio_')
test = test.merge(ev_ratios, on='cookie_id', how='left')


### Боты, в отличие от людей, почти не совершают действий помимо сбора данных, не пишут продавцам и т.д.

In [21]:
train['contact_per_view']  = train.get('ev_contact_phone_show', 0) / train['n_events'].replace(0, np.nan)
train['favorite_per_view'] = train.get('ev_favorite_add', 0) / train['n_events'].replace(0, np.nan)
train['login_per_view']    = train.get('ev_login', 0) / train['n_events'].replace(0, np.nan)

test['contact_per_view']  = test.get('ev_contact_phone_show', 0) / test['n_events'].replace(0, np.nan)
test['favorite_per_view'] = test.get('ev_favorite_add', 0) / test['n_events'].replace(0, np.nan)
test['login_per_view']    = test.get('ev_login', 0) / test['n_events'].replace(0, np.nan)

### Люди реже пользуются сайтом ночью, боты же работают 24/7. Поэтому выделим отдельно night_ratio.

In [22]:
ev_tr.columns

Index(['cookie_id', 'event_ts', 'eid', 'event_name', 'platform', 'user_agent',
       'item_id', 'item_category', 'item_location', 'seller_type',
       'search_query', 'search_page', 'pointer_x', 'pointer_y', 'ua_lower',
       'ua_is_python', 'ua_is_go', 'ua_is_curl', 'ua_os', 'ua_browser',
       'ua_is_mobile', 'ua_is_pc', 'window_start_ts', 'window_end_ts',
       'time_delta_sec'],
      dtype='object')

In [23]:
def add_night_features(ev):
    """
    Считает долю ночных событий и число уникальных часов внутри окна.
    """
    ev = ev.copy()
    ev['hour'] = ev['event_ts'].dt.hour

    night_ratio = (
        ((ev['hour'] >= 1) & (ev['hour'] <= 6))
        .groupby(ev['cookie_id'])
        .mean()
        .rename('night_ratio')
    )
    n_unique_hours = (
        ev.groupby('cookie_id')['hour']
          .nunique()
          .rename('n_unique_hours')
    )

    out = pd.concat([night_ratio, n_unique_hours], axis=1).reset_index()
    return out

night_tr = add_night_features(ev_tr)
train = train.merge(night_tr, on='cookie_id', how='left')

night_te = add_night_features(ev_te)
test = test.merge(night_te, on='cookie_id', how='left')

In [24]:
!ls

drive  sample_data


## **Train/valid/test**

In [25]:
Xtr = train.copy()
ytr = train.target.values
Xtr = Xtr.drop(columns=['target', 'cookie_id', 'cookie_created_at', 'window_start_ts', 'window_end_ts'])

Xte = test.copy()
Xte = Xte.drop(columns=['cookie_id', 'cookie_created_at', 'window_start_ts', 'window_end_ts'])

In [26]:
# Удалить редкие one-hot колонки
threshold = 0.01  # доля
freq = Xtr[Xtr.select_dtypes(exclude=['string', 'object']).columns.tolist()].mean()
rare_cols = freq[freq < threshold].index
Xtr = Xtr.drop(columns=rare_cols)
Xte = Xte.drop(columns=rare_cols)

### Итоговые фичи на нашем датасете

In [27]:
Xtr.columns

Index(['time_mean', 'time_max', 'time_min', 'time_std', 'time_median',
       'n_events', 'pointer_x_mean', 'pointer_x_max', 'pointer_x_min',
       'pointer_x_std', 'pointer_x_median', 'pointer_y_mean', 'pointer_y_max',
       'pointer_y_min', 'pointer_y_std', 'pointer_y_median',
       'pointer_total_distance', 'pointer_max_jump', 'n_unique_ua',
       'n_unique_query', 'max_search', 'mobile_ratio', 'pc_ratio',
       'mean_search', 'main_platform', 'main_os', 'main_browser',
       'ev_contact_chat_open', 'ev_contact_message_sent',
       'ev_contact_phone_show', 'ev_favorite_add', 'ev_item_view', 'ev_login',
       'ev_photo_swipe', 'ev_search_results_view', 'ev_seller_page_view',
       'n_unique_events', 'n_unique_items', 'item_diversity',
       'n_unique_categories', 'n_unique_locations', 'n_unique_sellers',
       'ratio_contact_chat_open', 'ratio_contact_phone_show',
       'ratio_favorite_add', 'ratio_item_view', 'ratio_login',
       'ratio_photo_swipe', 'ratio_search_resul

In [28]:
Xte = Xte[Xtr.columns]   # имена и порядок как в Xtr

# Контрольная проверка
assert list(Xte.columns) == list(Xtr.columns), 'Колонки Xtr и Xte не совпадают'

In [29]:
#категориальные колонки
cat_cols = Xtr.select_dtypes(include=['string', 'object']).columns.tolist()
cat_cols
for c in cat_cols:
    Xtr[c] = Xtr[c].fillna('unknown').astype(str)
    Xte[c] = Xte[c].fillna('unknown').astype(str)

### 1. Обучение с валидацией — для контроля метрики

In [31]:
from catboost import CatBoostClassifier
from metric import precision_at_recall   # официальная реализация, ей же считает автопроверка

is_valid = train.window_start_ts.ge('2026-04-17').values

model_cv = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=20,           # регуляризация
    auto_class_weights='Balanced',
    eval_metric='PRAUC',      # близко к вашей метрике
    early_stopping_rounds=50,
    verbose=100,
    use_best_model=False,
    random_seed=rs
)

model_cv.fit(Xtr.loc[~is_valid], ytr[~is_valid], eval_set=(Xtr.loc[is_valid], ytr[is_valid]), cat_features=cat_cols)
p_va = model_cv.predict_proba(Xtr.loc[is_valid])[:, 1]
print('P@R0.7 на валидации:', round(precision_at_recall(ytr[is_valid], p_va), 4))


0:	learn: 0.8440942	test: 0.8041725	best: 0.8041725 (0)	total: 73.5ms	remaining: 1m 13s
100:	learn: 0.9506132	test: 0.9265559	best: 0.9265559 (100)	total: 3s	remaining: 26.7s
200:	learn: 0.9664711	test: 0.9304441	best: 0.9304441 (200)	total: 6.5s	remaining: 25.8s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9315422045
bestIteration = 239

P@R0.7 на валидации: 0.7179


### 2. Финальная модель — на всех train-данных


### **Сдаём скоры для тестовой выборки**

In [32]:

final_iter = model_cv.get_best_iteration() or 500
model_final = CatBoostClassifier(
    iterations=final_iter,
    learning_rate=0.05, depth=6, l2_leaf_reg=20,
    auto_class_weights='Balanced',
    random_seed=rs, verbose=0,
)
model_final.fit(Xtr, ytr, cat_features=cat_cols)



CatBoostClassifier(auto_class_weights='Balanced', depth=6, iterations=239, l2_leaf_reg=20, learning_rate=0.05, random_seed=0, verbose=0)

In [33]:
# Предсказание
sub = pd.DataFrame({
    'cookie_id': test.cookie_id,
    'score': model_final.predict_proba(Xte)[:, 1],
})
assert len(sub) == len(test) and sub.score.between(0, 1).all()
sub.to_csv('submission.csv', index=False)
sub.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.015615
1,ck_a76ee3b3e3e522fd,0.561527
2,ck_94c9a4d382689e82,0.105282
3,ck_8eaf9509ad9462a0,0.045047
4,ck_9a88a5a989cb5bc6,0.049006


### **Feature importance**

In [34]:
imp = pd.Series(
    model_cv.get_feature_importance(),
    index=Xtr.columns
).sort_values(ascending=False)
print(imp.head(20))
print(f'\nПризнаков с importance < 0.005: {(imp < 0.005).sum()}')

time_median                  13.057082
pointer_x_std                11.308735
n_events                      5.816245
mean_search                   5.356884
pointer_x_max                 4.864502
time_min                      4.256735
ratio_photo_swipe             4.184548
ratio_item_view               3.125177
ratio_favorite_add            3.050723
pointer_max_jump              2.781977
ratio_contact_phone_show      2.656534
ratio_search_results_view     2.597263
max_search                    2.576891
time_mean                     2.454880
main_browser                  2.133967
n_unique_query                1.790210
main_os                       1.730417
time_max                      1.702998
favorite_per_view             1.658169
ratio_seller_page_view        1.626820
dtype: float64

Признаков с importance < 0.005: 5


## Выводы

### Что сработало
- Явные UA-сигнатуры (`python-requests`, `Go-http-client`, `curl`) — почти идеально делят ботов.
- Временные интервалы внутри окна — у ботов разброс времени близок к нулю.
- Pointer-статистики — у ботов курсор либо отсутствует, либо смещения аномально малы.
- Доли событий (`ratio_contact_*`, `ratio_favorite_add`) — у ботов конверсия близка к нулю.
- Ночная активность (`night_ratio`) — боты работают 24/7.

### Что не сработало / убрал
- One-hot по редким категориям браузеров давал переобучение → CatBoost с `cat_features`.
- Абсолютные счётчики без нормализации на длину окна плохо разделяли классы.

### Ограничения
- Валидация — один временной сплит (>= 2026-04-17). На нескольких сплитах
  ожидаю разброс P@R0.7 ±0.03.
- Все агрегаты на уровне `cookie_id`; при появлении нескольких окон на куку пайплайн
  нужно будет слегка изменить

### Куда развивать
- Ансамбль CatBoost + RandomForest.
- Target encoding (иногда даёт +1-2 пункта).